In [1]:
!mkdir -p /root/.kaggle

!cp "kaggle.json" /root/.kaggle/kaggle.json

!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d dasa7753912/glaucoma-detection

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/dasa7753912/glaucoma-detection
License(s): unknown
100% 568M/568M [00:27<00:00, 21.3MB/s]



In [2]:
!unzip glaucoma-detection.zip

Streaming output truncated to the last 5000 lines.
  inflating: Datasets/Rim-One/Glaucoma/train-38001.png  
  inflating: Datasets/Rim-One/Glaucoma/train-38002.png  
  inflating: Datasets/Rim-One/Glaucoma/train-38003.png  
  inflating: Datasets/Rim-One/Glaucoma/train-38004.png  
  inflating: Datasets/Rim-One/Glaucoma/train-38005.png  
  inflating: Datasets/Rim-One/Glaucoma/train-38006.png  
  inflating: Datasets/Rim-One/Glaucoma/train-38007.png  
  inflating: Datasets/Rim-One/Glaucoma/train-38008.png  
  inflating: Datasets/Rim-One/Glaucoma/train-38009.png  
  inflating: Datasets/Rim-One/Glaucoma/train-38010.png  
  inflating: Datasets/Rim-One/Glaucoma/train-38011.png  
  inflating: Datasets/Rim-One/Glaucoma/train-38012.png  
  inflating: Datasets/Rim-One/Glaucoma/train-38013.png  
  inflating: Datasets/Rim-One/Glaucoma/train-38014.png  
  inflating: Datasets/Rim-One/Glaucoma/train-38015.png  
  inflating: Datasets/Rim-One/Glaucoma/train-38016.png  
  inflating: Datasets/Rim-One/Glaucom

In [3]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

In [4]:
import os
import shutil
import random
from sklearn.model_selection import train_test_split

random.seed(42)

dataset_path = "/content/Datasets/Acrima"
output_path = "/content/Datasets/Acrima_Split"

classes = ["Glaucoma", "Normal"]

for cls in classes:

    images = os.listdir(os.path.join(dataset_path, cls))

    train_imgs, temp_imgs = train_test_split(
        images,
        test_size=0.30,
        random_state=42,
        shuffle=True
    )

    val_imgs, test_imgs = train_test_split(
        temp_imgs,
        test_size=0.50,
        random_state=42,
        shuffle=True
    )

    for split in ["Train", "Validation", "Test"]:
        os.makedirs(os.path.join(output_path, split, cls), exist_ok=True)

    for img in train_imgs:
        shutil.copy(
            os.path.join(dataset_path, cls, img),
            os.path.join(output_path, "Train", cls, img)
        )

    for img in val_imgs:
        shutil.copy(
            os.path.join(dataset_path, cls, img),
            os.path.join(output_path, "Validation", cls, img)
        )

    for img in test_imgs:
        shutil.copy(
            os.path.join(dataset_path, cls, img),
            os.path.join(output_path, "Test", cls, img)
        )

print("Dataset Split Completed Successfully!")

Dataset Split Completed Successfully!


In [5]:
import os

train_path = "/content/Datasets/Acrima_Split/Train"
val_path = "/content/Datasets/Acrima_Split/Validation"
test_path = "/content/Datasets/Acrima_Split/Test"

classes = ["Glaucoma", "Normal"]

for split, path in [("Train", train_path),
                    ("Validation", val_path),
                    ("Test", test_path)]:

    print(f"\n{split} Dataset")

    total = 0
    for cls in classes:
        count = len(os.listdir(os.path.join(path, cls)))
        total += count
        print(f"{cls}: {count}")

    print(f"Total Images: {total}")


Train Dataset
Glaucoma: 2100
Normal: 2100
Total Images: 4200

Validation Dataset
Glaucoma: 450
Normal: 450
Total Images: 900

Test Dataset
Glaucoma: 450
Normal: 450
Total Images: 900


In [6]:
from tensorflow.keras.applications.efficientnet import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    val_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

test_data = test_datagen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

print("Train:", train_data.samples)
print("Validation:", val_data.samples)
print("Test:", test_data.samples)

Found 4200 images belonging to 2 classes.
Found 900 images belonging to 2 classes.
Found 900 images belonging to 2 classes.
Train: 4200
Validation: 900
Test: 900


In [7]:
print(train_data.class_indices)

from collections import Counter

print(Counter(train_data.classes))

{'Glaucoma': 0, 'Normal': 1}
Counter({np.int32(0): 2100, np.int32(1): 2100})


In [8]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)


base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [9]:
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout,
    BatchNormalization
)

x = base_model.output

# Layer 1
x = GlobalAveragePooling2D()(x)

# Layer 2
x = Dense(512, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

# Layer 3
x = Dense(256, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

# Layer 4
x = Dense(128, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

# Layer 5
x = Dense(64, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

# Layer 6
x = Dense(32, activation="relu")(x)

# Layer 7
x = Dense(16, activation="relu")(x)

# Layer 8 (Output)
output = Dense(1, activation="sigmoid")(x)


model = Model(
    inputs=base_model.input,
    outputs=output
)

In [10]:
model.compile(
    optimizer=Adam(learning_rate=5e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,884,388 (18.63 MB)

 Trainable params: 2,329,057 (8.88 MB)

 Non-trainable params: 2,555,331 (9.75 MB)

In [11]:
model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 146s 776ms/step - accuracy: 0.5326 - loss: 0.7179 - val_accuracy: 0.5422 - val_loss: 0.6879
Epoch 2/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 60s 453ms/step - accuracy: 0.5912 - loss: 0.6666 - val_accuracy: 0.6778 - val_loss: 0.6756
Epoch 3/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 60s 451ms/step - accuracy: 0.6657 - loss: 0.6117 - val_accuracy: 0.7544 - val_loss: 0.6443
Epoch 4/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 58s 436ms/step - accuracy: 0.7148 - loss: 0.5611 - val_accuracy: 0.7700 - val_loss: 0.5952
Epoch 5/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 59s 449ms/step - accuracy: 0.7557 - loss: 0.5103 - val_accuracy: 0.7900 - val_loss: 0.5227
Epoch 6/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 59s 447ms/step - accuracy: 0.7874 - loss: 0.4660 - val_accuracy: 0.7856 - val_loss: 0.4780
Epoch 7/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 59s 448ms/step - accuracy: 0.8245 - loss: 0.4173 - val_accuracy: 0.8044 - val_loss: 0.4292
Epoch 8/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 60s 453ms/step - accuracy: 0.8462 - loss: 

In [12]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Reset test generator
test_data.reset()

# Predict
y_pred_prob = model.predict(test_data, verbose=1)

# Convert probabilities to 0 and 1
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# True labels
y_true = test_data.classes

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix")
print(cm)

print("\nClassification Report")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Glaucoma", "Normal"]
))

29/29 ━━━━━━━━━━━━━━━━━━━━ 21s 319ms/step
Confusion Matrix
[[446   4]
 [139 311]]

Classification Report
              precision    recall  f1-score   support

    Glaucoma       0.76      0.99      0.86       450
      Normal       0.99      0.69      0.81       450

    accuracy                           0.84       900
   macro avg       0.87      0.84      0.84       900
weighted avg       0.87      0.84      0.84       900



In [13]:
model.save('gloucoma_detection.h5')

In [15]:
from PIL import Image
import numpy as np
from tensorflow.keras.applications.efficientnet import preprocess_input

img = Image.open("/content/Datasets/Acrima_Split/Test/Normal/train-17846.png").convert("RGB")
img = img.resize((224,224))

img_array = np.array(img).astype(np.float32)
img_array = np.expand_dims(img_array, axis=0)

# Same preprocessing as training
img_array = preprocess_input(img_array)

# Prediction
pred = model.predict(img_array, verbose=0)[0][0]

print("Prediction Value:", pred)

if pred > 0.5:
    print("Normal")
else:
    print("Glaucoma")

Prediction Value: 0.7621439
Normal


In [16]:
import os

path = "/content/Datasets/Acrima_Split/Test/Normal/train-17846.png"

print(os.path.exists(path))
print(path)

True
/content/Datasets/Acrima_Split/Test/Normal/train-17846.png
